# GNM-Contact Learner: 2000 PDB Scale-Up Training

4-phase pipeline:
1. **PDB Curation** — RCSB API ile 2000 diverse protein
2. **Chunked Inference** — 10'arlık batch ile OpenFold3 pair repr extraction
3. **Shard Packing** — .pt → .npz (50/shard)
4. **Training** — Focal loss + OneCycleLR + gradient accumulation

Her cell resume-safe. Colab disconnect olursa kaldığın yerden devam eder.

## 1. Environment Setup

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════════════════
!pip install -q biopython requests wandb

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 2: Clone / Update Repository
# ══════════════════════════════════════════════════════════════════
import os
from pathlib import Path

REPO_DIR = Path('/content/ANM-openfold3')

if REPO_DIR.exists():
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/beratkaskaloglu/ANM-openfold3.git {REPO_DIR}

os.chdir(str(REPO_DIR))
print(f"Working dir: {os.getcwd()}")
!git log --oneline -5

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 3: Setup OpenFold3 (clone + install)
# ══════════════════════════════════════════════════════════════════
OF3_DIR = Path('/content/ANM-openfold3/openfold3-repo')

if not OF3_DIR.exists():
    !git clone https://github.com/aqlaboratory/openfold-3.git {OF3_DIR}

# Always install — subprocess scripts need it in site-packages
!pip install -e {OF3_DIR} -q

import sys
sys.path.insert(0, str(OF3_DIR))
sys.path.insert(0, str(REPO_DIR))

# Verify
try:
    from openfold3.entry_points.validator import InferenceExperimentConfig
    from openfold3.entry_points.experiment_runner import InferenceExperimentRunner
    print('OpenFold3 OK')
except ImportError as e:
    print(f'FAILED: {e}')

In [ ]:
# ══════════════════════════════════════════════════════════════════#  CELL 3b: Download OpenFold3 Weights##  OF3 model weights interaktif yes/no soruyor.#  Bu cell onlari onceden indiriyor.#  Zaten varsa (~/.openfold3/) skip eder.# ══════════════════════════════════════════════════════════════════from pathlib import Pathweights_path = Path.home() / ".openfold3" / "of3-p2-155k.pt"if weights_path.exists():    print(f"Weights already cached: {weights_path} ({weights_path.stat().st_size / 1e9:.1f} GB)")else:    print("Downloading OpenFold3 weights...")    !echo yes | python -c "import sys; sys.path.insert(0,'/content/ANM-openfold3/openfold3-repo'); from openfold3.entry_points.validator import InferenceExperimentConfig; from openfold3.entry_points.experiment_runner import InferenceExperimentRunner; config = InferenceExperimentConfig(); runner = InferenceExperimentRunner(config, num_diffusion_samples=1, num_model_seeds=1, use_msa_server=False, use_templates=False, output_dir='/tmp/of3_wt_test'); runner.setup(); print('OK')"    if weights_path.exists():        print(f"Downloaded: {weights_path} ({weights_path.stat().st_size / 1e9:.1f} GB)")    else:        print("WARNING: download may have failed")

## 2. Phase 1 — PDB Curation (2000 proteins)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 4: Fetch 2000 diverse PDBs from RCSB
#
#  Filters: X-ray, ≤2.5Å, 30-500 aa, ≤30% seq identity
#  Resume-safe: skips if data/pdb_2000.json already exists
# ══════════════════════════════════════════════════════════════════
import json

PDB_LIST = Path('data/pdb_2000.json')

if PDB_LIST.exists():
    proteins = json.loads(PDB_LIST.read_text())
    print(f"PDB list already exists: {len(proteins)} proteins")
else:
    !python scripts/fetch_pdb_list.py --target 2000
    proteins = json.loads(PDB_LIST.read_text())
    print(f"Fetched: {len(proteins)} proteins")

# Stats
lengths = [p['length'] for p in proteins]
print(f"Length: min={min(lengths)}, max={max(lengths)}, "
      f"mean={sum(lengths)/len(lengths):.0f}, "
      f"median={sorted(lengths)[len(lengths)//2]}")

## 3. Phase 2+3 — Chunked Inference + Inline Shard Packing

10 protein / inference chunk, 50 protein / .npz shard.

Her 50 protein'den sonra:
1. .pt → .npz shard'a pakle
2. .pt dosyalarını sil (disk max ~1-2 GB)

**~100s / protein → ~55 saat total.**
Resume-safe: tamamlanan shard'lar skip edilir.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5: Google Drive Mount + Directories
#
#  Drive'da sadece checkpoints ve progress tutuluyor (< 10 MB).
#  Shardlar local disk'te — egitildikten sonra silinebilir.
#  Session bitince progress sayesinde kaldigin yerden devam eder.
# ══════════════════════════════════════════════════════════════════
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Drive paths (kalici, < 10 MB)
DRIVE_BASE   = Path('/content/drive/MyDrive/ANM-openfold3')
CKPT_DIR     = DRIVE_BASE / 'checkpoints'
PROGRESS_DIR = DRIVE_BASE / 'progress'

# Local paths (hizli I/O, session sonunda silinir)
SHARD_DIR     = Path('/content/shards')
PAIR_REPR_DIR = Path('/content/pair_reprs')
COORDS_DIR    = Path('/content/coords')
OF3_OUTPUT    = Path('/content/of3_output')
QUERY_DIR     = Path('/content/queries')

for d in [CKPT_DIR, PROGRESS_DIR, SHARD_DIR,
          PAIR_REPR_DIR, COORDS_DIR, OF3_OUTPUT, QUERY_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  {d}")

# Resume durumu
n_done = len(list(PROGRESS_DIR.glob('shard_*.ok')))
print(f"\nTamamlanan batch: {n_done} ({n_done * 10} protein)")
if (CKPT_DIR / 'latest.pt').exists():
    print(f"Checkpoint: {CKPT_DIR / 'latest.pt'}")
else:
    print("Checkpoint: yok (ilk calisma)")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 6: Incremental Inference + Inline Training Loop
#
#  full_pipeline.ipynb'deki calisan yaklasim:
#  - Training inline (subprocess degil)
#  - Dim handling acik (squeeze/unsqueeze kontrol)
#  - Hatalar hemen gorunur
# ══════════════════════════════════════════════════════════════════
import os, json, time, sys
import numpy as np
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

sys.path.insert(0, '/content/ANM-openfold3')
from src.contact_head import ContactProjectionHead
from src.data import ShardedPairReprDataset
from src.losses import total_loss
from src.kirchhoff import soft_kirchhoff, gnm_decompose

# === CONFIG ===
BATCH_SIZE = 100
TRAIN_EPOCHS = 50          # 30 -> 50: daha fazla data ile daha fazla epoch
TOTAL_PROTEINS = 1000
LR = 1e-4                  # 3e-4 -> 1e-4: daha stabil ogrenme
WEIGHT_DECAY = 1e-2
BOTTLENECK_DIM = 64
C_Z = 128
R_CUT = 8.0
TAU = 1.0
ALPHA = 1.0
BETA = 0.3
GAMMA = 0.05
N_MODES = 20
GRAD_CLIP = 1.0
SHARD_KEEP = 5              # son 5 shard tut (~50 protein)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def validate_shards(shard_dir):
    """Check all shards are readable, delete corrupt ones."""
    for f in sorted(shard_dir.glob('shard_*.npz')):
        try:
            with np.load(f, allow_pickle=True) as data:
                ids = data['pdb_ids']
                # Verify at least one protein has data
                if len(ids) == 0 or 'pair_repr_0' not in data:
                    raise ValueError(f"Empty shard: {len(ids)} ids")
        except Exception as e:
            print(f"[Cleanup] Corrupt shard {f.name}: {e}")
            f.unlink()

def inline_train(shard_dir, ckpt_dir, epochs, device):
    """Train inline — same approach as full_pipeline.ipynb."""
    shard_paths = sorted(shard_dir.glob('shard_*.npz'))
    if not shard_paths:
        print("[Train] No shards found, skipping")
        return

    dataset = ShardedPairReprDataset(shard_paths, r_cut=R_CUT, tau=TAU)
    n_total = len(dataset)
    if n_total < 2:
        print(f"[Train] Only {n_total} proteins, need >= 2, skipping")
        return

    n_val = max(1, int(n_total * 0.15))
    n_train = n_total - n_val
    train_ds, val_ds = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

    # Model
    head = ContactProjectionHead(c_z=C_Z, bottleneck_dim=BOTTLENECK_DIM).to(device)

    # Resume weights if checkpoint exists
    ckpt_path = ckpt_dir / 'latest.pt'
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        head.load_state_dict(ckpt['model_state_dict'])
        print(f"[Train] Resumed weights (prev val_loss={ckpt.get('val_loss', '?')})")

    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    print(f"[Train] {n_train} train, {n_val} val, {epochs} epochs, device={device}")

    for epoch in range(epochs):
        # ── Train ──
        head.train()
        train_loss_sum = 0.0
        train_n = 0

        for batch in train_loader:
            z = batch['pair_repr'].to(device)
            c_gt = batch['c_gt'].to(device)

            # Handle dimensions (full_pipeline approach)
            if z.dim() == 5:
                z = z.squeeze(0)   # [1,1,N,N,c_z] -> [1,N,N,c_z]
            if c_gt.dim() == 3:
                c_gt = c_gt.squeeze(0)  # [1,N,N] -> [N,N]

            c_pred = head(z).squeeze(0)  # [N, N]

            # Reconstruction
            z_sym = 0.5 * (z + z.transpose(1, 2))
            h = head.w_enc(z_sym)
            z_recon = head.w_dec(h)

            loss, details = total_loss(
                c_pred, c_gt,
                z_original=z_sym, z_reconstructed=z_recon,
                alpha=ALPHA, beta=BETA, gamma=GAMMA, n_modes=N_MODES,
                use_focal=True, focal_gamma=2.0, focal_alpha=0.75,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP)
            optimizer.step()

            train_loss_sum += loss.item()
            train_n += 1

        scheduler.step()
        avg_train = train_loss_sum / max(train_n, 1)

        # ── Validate ──
        head.eval()
        val_loss_sum = 0.0
        val_n = 0

        with torch.no_grad():
            for batch in val_loader:
                z = batch['pair_repr'].to(device)
                c_gt = batch['c_gt'].to(device)
                if z.dim() == 5:
                    z = z.squeeze(0)
                if c_gt.dim() == 3:
                    c_gt = c_gt.squeeze(0)
                c_pred = head(z).squeeze(0)
                loss, _ = total_loss(
                    c_pred, c_gt, alpha=ALPHA, beta=BETA,
                    n_modes=N_MODES, use_focal=True,
                )
                val_loss_sum += loss.item()
                val_n += 1

        avg_val = val_loss_sum / max(val_n, 1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1}/{epochs}  train={avg_train:.4f}  val={avg_val:.4f}")

        # Save best
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save({
                'model_state_dict': head.state_dict(),
                'epoch': epoch,
                'val_loss': best_val_loss,
                'c_z': C_Z,
                'bottleneck_dim': BOTTLENECK_DIM,
            }, ckpt_dir / 'best_model.pt')

    # Always save latest
    torch.save({
        'model_state_dict': head.state_dict(),
        'epoch': epoch,
        'val_loss': avg_val,
        'c_z': C_Z,
        'bottleneck_dim': BOTTLENECK_DIM,
    }, ckpt_dir / 'latest.pt')

    print(f"[Train] Done! best_val={best_val_loss:.4f}")
    return best_val_loss

# === Resume ===
# Progress files are from old batch sizes (10/batch), count actual proteins
existing_progress = sorted(PROGRESS_DIR.glob('shard_*.ok'))
DONE_PROTEINS = len(existing_progress) * 10  # each old .ok = 10 proteins
START_BATCH = DONE_PROTEINS // BATCH_SIZE     # align to new batch size
START_PROTEIN = START_BATCH * BATCH_SIZE
print(f"Tamamlanan: {len(existing_progress)} eski batch ({DONE_PROTEINS} protein)")
print(f"Yeni batch size={BATCH_SIZE}, devam: batch {START_BATCH} (protein {START_PROTEIN})")

# === LOOP ===
n_batches = (TOTAL_PROTEINS + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(START_BATCH, n_batches):
    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, TOTAL_PROTEINS)

    print(f"\n{'='*60}")
    print(f"  BATCH {batch_idx}: proteins {start}-{end}")
    print(f"{'='*60}")

    # 1. Inference
    t0 = time.time()
    infer_cmd = (
        f'echo yes | python scripts/extract_pairs.py'
        f' --pdb-list data/pdb_2000.json'
        f' --shard-size {BATCH_SIZE}'
        f' --inference-chunk-size {BATCH_SIZE}'
        f' --start {start} --end {end}'
        f' --pair-repr-dir {PAIR_REPR_DIR}'
        f' --coords-dir {COORDS_DIR}'
        f' --output-dir {OF3_OUTPUT}'
        f' --query-dir {QUERY_DIR}'
        f' --shard-dir {SHARD_DIR}'
        f' --progress-dir {PROGRESS_DIR}'
    )
    ret = os.system(infer_cmd)
    t_infer = time.time() - t0
    print(f"[Inference] done in {t_infer:.0f}s (exit={ret})")

    # 2. Validate + cleanup shards (keep last SHARD_KEEP)
    validate_shards(SHARD_DIR)
    old_shards = sorted(SHARD_DIR.glob('shard_*.npz'))[:-SHARD_KEEP]
    for f in old_shards:
        f.unlink()
        print(f"[Cleanup] Deleted {f.name}")

    # 3. Inline training
    t0 = time.time()
    try:
        inline_train(SHARD_DIR, CKPT_DIR, TRAIN_EPOCHS, DEVICE)
    except Exception as e:
        print(f"[Train ERROR] {e}")
    t_train = time.time() - t0
    print(f"[Train] {t_train:.0f}s")

    # 4. Status
    n_shards = len(list(SHARD_DIR.glob('shard_*.npz')))
    n_progress = len(list(PROGRESS_DIR.glob('shard_*.ok')))
    ckpt_ok = (CKPT_DIR / 'latest.pt').exists()
    best_ok = (CKPT_DIR / 'best_model.pt').exists()
    print(f"[Status] Progress: {n_progress} batches ({n_progress * BATCH_SIZE} proteins)")
    print(f"[Status] latest.pt: {'YES' if ckpt_ok else 'NO'}, "
          f"best_model.pt: {'YES' if best_ok else 'NO'}")

    # 5. Free GPU memory
    torch.cuda.empty_cache()

print("\n=== ALL DONE ===")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7: Progress Check
# ══════════════════════════════════════════════════════════════════
import json

ok_files = sorted(PROGRESS_DIR.glob('shard_*.ok'))
total_ok = 0
total_fail = 0
all_failed = []

for f in ok_files:
    info = json.loads(f.read_text())
    total_ok += info['success']
    total_fail += len(info.get('failed', []))
    all_failed.extend(info.get('failed', []))

n_shards = len(list(SHARD_DIR.glob('shard_*.npz')))
total_size = sum(f.stat().st_size for f in SHARD_DIR.glob('shard_*.npz')) / 1e6

print(f'Shards: {n_shards} files ({total_size:.0f} MB)')
print(f'Success: {total_ok}, Failed: {total_fail}')
if all_failed:
    print(f'Failed: {all_failed[:20]}{"..." if len(all_failed) > 20 else ""}')

## 4. Phase 4 — Training

Shard packing artik inference ile birlikte yapiliyor.
Direkt training'e gecebilirsin.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 8: Verify Shards
# ══════════════════════════════════════════════════════════════════
import numpy as np

shard_files = sorted(SHARD_DIR.glob('shard_*.npz'))
total_proteins = 0
for sf in shard_files:
    data = np.load(sf, allow_pickle=True)
    n = len(data['pdb_ids'])
    total_proteins += n

print(f'Total shards: {len(shard_files)}')
print(f'Total proteins: {total_proteins}')
print(f'Ready for training!')

## 5. Training

**Hyperparameters:**
- Focal Loss (gamma=2.0, alpha=0.75)
- OneCycleLR (warmup 5% + cosine decay)
- Gradient accumulation (4 steps)
- r_cut=8.0 A, tau=1.0
- bottleneck_dim=64
- Early stopping (patience=50)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 9: Kernel restart gerekirse — sadece bu cell'i çalıştır
#  GPU belleği temizlemek için:
# ══════════════════════════════════════════════════════════════════
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 10: Fine-Tuning — Tum veriye uzun training
#
#  Incremental loop bitti (~900 protein).
#  Simdi tum shard'lar uzerinde dedicated fine-tuning.
# ══════════════════════════════════════════════════════════════════
import sys, time, json
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Subset

sys.path.insert(0, '/content/ANM-openfold3')
from src.contact_head import ContactProjectionHead
from src.data import ShardedPairReprDataset
from src.losses import total_loss
from src.kirchhoff import soft_kirchhoff, gnm_decompose

# === FINE-TUNE CONFIG ===
FT_EPOCHS = 500
FT_LR = 5e-5             # incremental'dan dusuk
FT_WEIGHT_DECAY = 1e-2
FT_PATIENCE = 60          # early stopping
FT_GRAD_CLIP = 1.0
FT_GRAD_ACCUM = 4         # effective batch = 4
FT_LOG_EVERY = 10
FT_SAVE_EVERY = 50
C_Z = 128
BOTTLENECK_DIM = 64
R_CUT = 8.0
TAU = 1.0
ALPHA = 1.0
BETA = 0.3
GAMMA = 0.05
N_MODES = 20
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# === Load all shards ===
shard_paths = sorted(SHARD_DIR.glob('shard_*.npz'))
print(f"Shards: {len(shard_paths)}")

dataset = ShardedPairReprDataset(shard_paths, r_cut=R_CUT, tau=TAU)
n_total = len(dataset)
print(f"Total proteins: {n_total}")

# Train / Val / Test split (80/10/10)
indices = list(range(n_total))
rng = np.random.RandomState(42)
rng.shuffle(indices)
n_test = int(n_total * 0.1)
n_val = int(n_total * 0.1)
n_train = n_total - n_val - n_test

train_ds = Subset(dataset, indices[:n_train])
val_ds = Subset(dataset, indices[n_train:n_train+n_val])
test_ds = Subset(dataset, indices[n_train+n_val:])
print(f"Split: train={n_train}, val={n_val}, test={n_test}")

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# === Model ===
head = ContactProjectionHead(c_z=C_Z, bottleneck_dim=BOTTLENECK_DIM).to(DEVICE)

# Resume from incremental training weights
ckpt_path = CKPT_DIR / 'best_model.pt'
if not ckpt_path.exists():
    ckpt_path = CKPT_DIR / 'latest.pt'
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    head.load_state_dict(ckpt['model_state_dict'])
    print(f"Resumed from {ckpt_path.name} (val_loss={ckpt.get('val_loss', '?')})")

n_params = sum(p.numel() for p in head.parameters())
print(f"Parameters: {n_params:,}")

# === Optimizer + Scheduler ===
optimizer = torch.optim.AdamW(head.parameters(), lr=FT_LR, weight_decay=FT_WEIGHT_DECAY)
total_steps = FT_EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=FT_LR, total_steps=total_steps,
    pct_start=0.05, anneal_strategy='cos',
    div_factor=10.0, final_div_factor=100.0,
)

# === Helpers ===
def pearson_corr(x, y):
    xc, yc = x - x.mean(), y - y.mean()
    return ((xc * yc).sum() / (xc.norm() * yc.norm()).clamp(min=1e-8)).item()

def adj_accuracy(c_pred, c_gt, thr=0.5):
    return ((c_pred > thr).float() == (c_gt > thr).float()).float().mean().item()

# === Training Loop ===
history = []
best_val_loss = float('inf')
epochs_no_improve = 0
ft_ckpt_dir = CKPT_DIR / 'finetune'
ft_ckpt_dir.mkdir(exist_ok=True)

print(f"\n{'='*60}")
print(f"  FINE-TUNING: {FT_EPOCHS} epochs, lr={FT_LR}, {n_train} proteins")
print(f"  grad_accum={FT_GRAD_ACCUM}, patience={FT_PATIENCE}")
print(f"{'='*60}\n")

t0_all = time.time()

for epoch in range(FT_EPOCHS):
    # ── Train ──
    head.train()
    train_loss_sum, train_n = 0.0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        z = batch['pair_repr'].to(DEVICE)
        c_gt = batch['c_gt'].to(DEVICE)
        if z.dim() == 5: z = z.squeeze(0)
        if c_gt.dim() == 3: c_gt = c_gt.squeeze(0)

        c_pred = head(z).squeeze(0)

        z_sym = 0.5 * (z + z.transpose(1, 2))
        h = head.w_enc(z_sym)
        z_recon = head.w_dec(h)

        loss, details = total_loss(
            c_pred, c_gt,
            z_original=z_sym, z_reconstructed=z_recon,
            alpha=ALPHA, beta=BETA, gamma=GAMMA, n_modes=N_MODES,
            use_focal=True, focal_gamma=2.0, focal_alpha=0.75,
        )

        (loss / FT_GRAD_ACCUM).backward()

        if (step + 1) % FT_GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(head.parameters(), FT_GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        train_loss_sum += loss.item()
        train_n += 1

    avg_train = train_loss_sum / max(train_n, 1)

    # ── Validate ──
    head.eval()
    val_loss_sum, val_acc_sum, val_bf_sum, val_n = 0.0, 0.0, 0.0, 0

    with torch.no_grad():
        for batch in val_loader:
            z = batch['pair_repr'].to(DEVICE)
            c_gt = batch['c_gt'].to(DEVICE)
            if z.dim() == 5: z = z.squeeze(0)
            if c_gt.dim() == 3: c_gt = c_gt.squeeze(0)

            c_pred = head(z).squeeze(0)
            loss, _ = total_loss(
                c_pred, c_gt, alpha=ALPHA, beta=BETA,
                n_modes=N_MODES, use_focal=True,
            )

            acc = adj_accuracy(c_pred, c_gt)
            gamma_p = soft_kirchhoff(c_pred)
            gamma_g = soft_kirchhoff(c_gt)
            _, _, bf_p = gnm_decompose(gamma_p, N_MODES)
            _, _, bf_g = gnm_decompose(gamma_g, N_MODES)
            bf_r = pearson_corr(bf_p, bf_g)

            val_loss_sum += loss.item()
            val_acc_sum += acc
            val_bf_sum += bf_r
            val_n += 1

    avg_val = val_loss_sum / max(val_n, 1)
    avg_acc = val_acc_sum / max(val_n, 1)
    avg_bf = val_bf_sum / max(val_n, 1)
    lr_now = optimizer.param_groups[0]['lr']

    record = {
        'epoch': epoch + 1,
        'train_loss': avg_train,
        'val_loss': avg_val,
        'val_adj_acc': avg_acc,
        'val_bf_pearson': avg_bf,
        'lr': lr_now,
    }
    history.append(record)

    # Log
    if (epoch + 1) % FT_LOG_EVERY == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{FT_EPOCHS}  "
              f"train={avg_train:.4f}  val={avg_val:.4f}  "
              f"acc={avg_acc:.3f}  bf_r={avg_bf:.3f}  lr={lr_now:.1e}")

    # Best model
    if avg_val < best_val_loss - 1e-5:
        best_val_loss = avg_val
        epochs_no_improve = 0
        torch.save({
            'model_state_dict': head.state_dict(),
            'epoch': epoch,
            'val_loss': best_val_loss,
            'val_adj_acc': avg_acc,
            'val_bf_pearson': avg_bf,
            'c_z': C_Z,
            'bottleneck_dim': BOTTLENECK_DIM,
        }, ft_ckpt_dir / 'best_model.pt')
    else:
        epochs_no_improve += 1

    # Save periodic + latest
    if (epoch + 1) % FT_SAVE_EVERY == 0:
        torch.save({
            'model_state_dict': head.state_dict(),
            'epoch': epoch, 'val_loss': avg_val,
            'c_z': C_Z, 'bottleneck_dim': BOTTLENECK_DIM,
        }, ft_ckpt_dir / f'epoch_{epoch+1:04d}.pt')

    torch.save({
        'model_state_dict': head.state_dict(),
        'epoch': epoch, 'val_loss': avg_val,
        'c_z': C_Z, 'bottleneck_dim': BOTTLENECK_DIM,
    }, ft_ckpt_dir / 'latest.pt')

    # Early stopping
    if epochs_no_improve >= FT_PATIENCE:
        print(f"\n  Early stopping at epoch {epoch+1} (no improve for {FT_PATIENCE} epochs)")
        break

# === Save history ===
(ft_ckpt_dir / 'history.json').write_text(json.dumps(history, indent=2))
total_time = time.time() - t0_all

# === Test evaluation ===
print(f"\n{'='*60}")
print(f"  FINE-TUNING COMPLETE")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"{'='*60}")

best_ckpt = torch.load(ft_ckpt_dir / 'best_model.pt', map_location=DEVICE, weights_only=False)
head.load_state_dict(best_ckpt['model_state_dict'])
head.eval()

test_loss_sum, test_acc_sum, test_bf_sum, test_n = 0.0, 0.0, 0.0, 0
with torch.no_grad():
    for batch in test_loader:
        z = batch['pair_repr'].to(DEVICE)
        c_gt = batch['c_gt'].to(DEVICE)
        if z.dim() == 5: z = z.squeeze(0)
        if c_gt.dim() == 3: c_gt = c_gt.squeeze(0)

        c_pred = head(z).squeeze(0)
        loss, _ = total_loss(c_pred, c_gt, alpha=ALPHA, beta=BETA, n_modes=N_MODES, use_focal=True)
        acc = adj_accuracy(c_pred, c_gt)
        gamma_p = soft_kirchhoff(c_pred)
        gamma_g = soft_kirchhoff(c_gt)
        _, _, bf_p = gnm_decompose(gamma_p, N_MODES)
        _, _, bf_g = gnm_decompose(gamma_g, N_MODES)
        bf_r = pearson_corr(bf_p, bf_g)

        test_loss_sum += loss.item()
        test_acc_sum += acc
        test_bf_sum += bf_r
        test_n += 1

test_metrics = {
    'loss': test_loss_sum / max(test_n, 1),
    'adj_acc': test_acc_sum / max(test_n, 1),
    'bf_pearson': test_bf_sum / max(test_n, 1),
}

print(f"\n  TEST RESULTS:")
print(f"    Loss:          {test_metrics['loss']:.4f}")
print(f"    Adj accuracy:  {test_metrics['adj_acc']:.4f}")
print(f"    B-factor r:    {test_metrics['bf_pearson']:.4f}")

test_results = {
    'test_metrics': test_metrics,
    'best_epoch': best_ckpt['epoch'] + 1,
    'best_val_loss': float(best_ckpt['val_loss']),
    'total_epochs': epoch + 1,
    'total_time_min': total_time / 60,
    'n_train': n_train, 'n_val': n_val, 'n_test': n_test,
}
(ft_ckpt_dir / 'test_results.json').write_text(json.dumps(test_results, indent=2))

## 6. Results & Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 11: Fine-Tuning Training Curves
# ══════════════════════════════════════════════════════════════════
import json
import matplotlib.pyplot as plt

ft_ckpt_dir = CKPT_DIR / 'finetune'
history = json.loads((ft_ckpt_dir / 'history.json').read_text())

epochs = [h['epoch'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss = [h['val_loss'] for h in history]
val_acc = [h['val_adj_acc'] for h in history]
val_bf = [h['val_bf_pearson'] for h in history]
lrs = [h['lr'] for h in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Fine-Tuning Results ({len(history)} epochs)', fontsize=14, fontweight='bold')

# Loss
axes[0,0].plot(epochs, train_loss, label='Train (contact+gnm+recon)', alpha=0.7)
axes[0,0].plot(epochs, val_loss, label='Val (contact+gnm)', alpha=0.7, linewidth=2)
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Loss')
axes[0,0].set_title('Loss Curves')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Adjacency Accuracy
axes[0,1].plot(epochs, val_acc, color='green', linewidth=2)
axes[0,1].axhline(y=0.85, color='red', linestyle='--', alpha=0.5, label='Target 0.85')
axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('Accuracy')
axes[0,1].set_title('Val Adjacency Accuracy')
axes[0,1].set_ylim(0.5, 1.0)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# B-factor correlation
axes[1,0].plot(epochs, val_bf, color='orange', linewidth=2)
axes[1,0].axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Target 0.70')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Pearson r')
axes[1,0].set_title('Val B-factor Correlation')
axes[1,0].set_ylim(0.0, 1.0)
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Learning rate
axes[1,1].plot(epochs, lrs, color='red', linewidth=2)
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('LR')
axes[1,1].set_title('OneCycleLR Schedule')
axes[1,1].set_yscale('log')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(ft_ckpt_dir / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print summary
best_epoch = min(history, key=lambda h: h['val_loss'])
print(f"Best epoch: {best_epoch['epoch']}")
print(f"  val_loss:    {best_epoch['val_loss']:.4f}")
print(f"  adj_acc:     {best_epoch['val_adj_acc']:.3f}")
print(f"  bf_pearson:  {best_epoch['val_bf_pearson']:.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 12: Test Set Results
# ══════════════════════════════════════════════════════════════════
import json

ft_ckpt_dir = CKPT_DIR / 'finetune'
test_results = json.loads((ft_ckpt_dir / 'test_results.json').read_text())

print("=" * 50)
print("  FINE-TUNING TEST SET RESULTS")
print("=" * 50)
print(f"  Best epoch:      {test_results['best_epoch']}")
print(f"  Best val loss:   {test_results['best_val_loss']:.4f}")
print(f"  Total epochs:    {test_results['total_epochs']}")
print(f"  Training time:   {test_results['total_time_min']:.1f} min")
print(f"  Data split:      {test_results['n_train']}/"
      f"{test_results['n_val']}/{test_results['n_test']}")
print()
m = test_results['test_metrics']
print(f"  Test loss:       {m['loss']:.4f}")
print(f"  Adj accuracy:    {m['adj_acc']:.4f}")
print(f"  B-factor r:      {m['bf_pearson']:.4f}")
print("=" * 50)

# Target check
targets = {'adj_acc': 0.85, 'bf_pearson': 0.70}
for k, target in targets.items():
    val = m[k]
    status = 'PASS' if val >= target else 'MISS'
print(f"  [{status}] {k}: {val:.4f} (target >= {target})")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 13: Contact Map Visualization
# ══════════════════════════════════════════════════════════════════
import sys, numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '/content/ANM-openfold3')

from src.contact_head import ContactProjectionHead
from src.data import ShardedPairReprDataset
from src.kirchhoff import soft_kirchhoff, gnm_decompose

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ft_ckpt_dir = CKPT_DIR / 'finetune'

# Load best model
ckpt = torch.load(ft_ckpt_dir / 'best_model.pt', map_location=DEVICE, weights_only=False)
head = ContactProjectionHead(c_z=ckpt['c_z'], bottleneck_dim=ckpt['bottleneck_dim']).to(DEVICE)
head.load_state_dict(ckpt['model_state_dict'])
head.eval()
print(f"Loaded best model (epoch {ckpt['epoch']+1}, val_loss={ckpt['val_loss']:.4f})")

# Load dataset
shard_paths = sorted(SHARD_DIR.glob('shard_*.npz'))
dataset = ShardedPairReprDataset(shard_paths, r_cut=8.0, tau=1.0)

# Show 6 random samples
n_show = min(6, len(dataset))
rng = np.random.RandomState(123)
show_idx = rng.choice(len(dataset), n_show, replace=False)

fig, axes = plt.subplots(n_show, 3, figsize=(12, 3.5 * n_show))
if n_show == 1:
    axes = axes[None, :]

with torch.no_grad():
    for row, idx in enumerate(show_idx):
        sample = dataset[idx]
        z = sample['pair_repr'].to(DEVICE)
        c_gt = sample['c_gt'].squeeze().cpu().numpy()
        c_pred = head(z).squeeze().cpu().numpy()
        pdb_id = sample.get('pdb_id', f'protein_{idx}')

        acc = ((c_pred > 0.5) == (c_gt > 0.5)).mean()

        axes[row, 0].imshow(c_gt, cmap='hot', vmin=0, vmax=1)
        axes[row, 0].set_title(f'{pdb_id} — Ground Truth')
        axes[row, 0].set_ylabel(f'N={c_gt.shape[0]}')

        axes[row, 1].imshow(c_pred, cmap='hot', vmin=0, vmax=1)
        axes[row, 1].set_title(f'{pdb_id} — Predicted (acc={acc:.2f})')

        diff = np.abs(c_pred - c_gt)
        axes[row, 2].imshow(diff, cmap='Blues', vmin=0, vmax=0.5)
        axes[row, 2].set_title(f'{pdb_id} — |Diff| (mae={diff.mean():.3f})')

plt.tight_layout()
plt.savefig(str(ft_ckpt_dir / 'contact_maps.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 14: B-factor Profile Comparison
# ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
N_MODES = 20

def pearson_corr(x, y):
    xc, yc = x - x.mean(), y - y.mean()
    return ((xc * yc).sum() / (xc.norm() * yc.norm()).clamp(min=1e-8)).item()

n_show = min(6, len(dataset))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

with torch.no_grad():
    for i, idx in enumerate(show_idx[:n_show]):
        sample = dataset[idx]
        z = sample['pair_repr'].to(DEVICE)
        c_gt = sample['c_gt'].squeeze().to(DEVICE)
        c_pred = head(z).squeeze()
        pdb_id = sample.get('pdb_id', f'protein_{idx}')

        _, _, bf_gt = gnm_decompose(soft_kirchhoff(c_gt), N_MODES)
        _, _, bf_pred = gnm_decompose(soft_kirchhoff(c_pred), N_MODES)

        bf_gt_n = (bf_gt / bf_gt.max()).cpu().numpy()
        bf_pred_n = (bf_pred / bf_pred.max()).cpu().numpy()
        r = pearson_corr(bf_pred.cpu(), bf_gt.cpu())

        axes[i].plot(bf_gt_n, label='GT', linewidth=2, color='#2196F3')
        axes[i].plot(bf_pred_n, label='Pred', linewidth=2, linestyle='--', color='#FF5722')
        axes[i].fill_between(range(len(bf_gt_n)), bf_gt_n, bf_pred_n, alpha=0.15, color='gray')
        axes[i].set_title(f'{pdb_id}  (r={r:.3f})')
        axes[i].set_xlabel('Residue')
        axes[i].set_ylabel('Norm. B-factor')
        axes[i].legend(fontsize=8)
        axes[i].grid(True, alpha=0.3)

# Hide unused axes
for j in range(n_show, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('B-factor Profiles: Ground Truth vs Predicted', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ft_ckpt_dir / 'bfactor_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Export & Download

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 15: Download Weights & Results
# ══════════════════════════════════════════════════════════════════
import shutil
from pathlib import Path

ft_ckpt_dir = CKPT_DIR / 'finetune'

# Pack everything into zip
zip_path = '/content/gnm_finetune_results'
shutil.make_archive(
    zip_path, 'zip',
    root_dir=str(ft_ckpt_dir.parent),
    base_dir=ft_ckpt_dir.name,
)
zip_size = Path(zip_path + '.zip').stat().st_size / 1e6
print(f"Packed: {zip_path}.zip ({zip_size:.1f} MB)")
print()

# Contents
print("ZIP contents:")
print("  finetune/")
print("    best_model.pt      — best val loss weights")
print("    latest.pt          — final epoch weights")
print("    history.json       — epoch-by-epoch metrics")
print("    test_results.json  — test set evaluation")
print("    training_curves.png")
print("    contact_maps.png")
print("    bfactor_profiles.png")
print()

# Also save best weights separately (smaller download)
best_only_path = '/content/best_model.pt'
shutil.copy(ft_ckpt_dir / 'best_model.pt', best_only_path)
best_size = Path(best_only_path).stat().st_size / 1e6
print(f"Best model only: {best_only_path} ({best_size:.1f} MB)")
print()

# Download
try:
    from google.colab import files
    print("Downloading zip...")
    files.download(f'{zip_path}.zip')
    print("\nDownloading best_model.pt separately...")
    files.download(best_only_path)
except ImportError:
    print("Not in Colab — download manually from file browser")

# Also copy to Drive as backup
try:
    drive_dest = Path('/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune')
    drive_dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(f'{zip_path}.zip', drive_dest / 'gnm_finetune_results.zip')
    shutil.copy(ft_ckpt_dir / 'best_model.pt', drive_dest / 'best_model.pt')
    print(f"\nBackup to Drive: {drive_dest}")
except Exception as e:
    print(f"Drive backup skipped: {e}")